### Numpy and Vectorization

#### Import necessary libaries and load dataset

In [1]:
import pandas as pd
import numpy as np
import time

weather = pd.read_csv('../data/raw/weather_daily.csv', 
                            na_values=['NA', ''])

#### Compute Evapotranspiration Using Ordinary Python Loops

In [2]:
weather['ET_loop'] = 0.0  

start_time_loop = time.time()

for i in range(len(weather)):
    temp  = weather.loc[i, 'temperature_c']
    wind  = weather.loc[i, 'wind_speed_mps']
    solar = weather.loc[i, 'solar_index']
    humid = weather.loc[i, 'humidity_pct']
    
    et_val = 0.12 * temp + 0.35 * wind + 2.4 * solar - 0.025 * humid
    weather.loc[i, 'ET_loop'] = max(0, et_val)

end_time_loop = time.time()
loop_execution_time = end_time_loop - start_time_loop
print(f"Loop execution time: {loop_execution_time:.4f} seconds")

Loop execution time: 0.0085 seconds


#### Compute Evapotranspiration Using NumPy Vectorization

In [4]:
start_time_vec = time.time()

et_vec = (0.12 * weather['temperature_c'] + 0.35 * weather['wind_speed_mps'] 
+ 2.4 * weather['solar_index'] - 0.025 * weather['humidity_pct'])

weather['ET_vec'] = np.maximum(0, et_vec)

end_time_vec = time.time()

vec_execution_time = end_time_vec - start_time_vec
print(f"Vectorized Execution Time: {vec_execution_time:.4f} seconds")

Vectorized Execution Time: 0.0017 seconds


#### Compare Execution Time Between Loop-based and Vectorized Computation

In [11]:
speedup = loop_execution_time / vec_execution_time
print(f"Ordinary Loop: {loop_execution_time:.5f} seconds")
print(f"Vectorized:    {vec_execution_time:.5f} seconds")
print(f"Vectorization is {speedup:.1f}x faster than standard Python loops.")

Ordinary Loop: 0.00848 seconds
Vectorized:    0.00173 seconds
Vectorization is 4.9x faster than standard Python loops.


#### Why Vectorization is Faster
1. **No Python overhead** - every iteration of a Python `for` loop has overhead i.e.
checking the loop counter, fetching the next index, updating variables etc. With 30
rows this is small, but with millions of rows it adds up massively. Vectorization does
it all in one operation with no per row Python overhead
2. **Operations run in C, not Python** - NumPy is written in C under the hood. It hands
the operations to a pre-compiled C routine which runs faster than interpreted Python
code
3. **Single Instruction, Multiple Data** - Modern CPUs process multiple numbers in a 
single instruction. NumPy takes advantage of this, so your CPU can multiply say 4 by
8 values simultaneously while a Python loop processes one value at a time.
4. **Better Memory Access Patterns** - NumPy stores data in contigious blocks of memory 
hence the CPU can load them efficiently in chunks. A Python loop accesses data through
objects and references scattered in memory, which is much slower to retrieve

Using vectorization is hence a foundational practice in scientific computing when scaling
large, real-time agro-meteorological data streams.



### Floating Point Errors and Numerical Reliability

#### Demonstrating Floating Point Behavior

Computers use the **IEEE 754 standard** to represent floating point numbers in binary.
Because base-2 cannot perfectly represent certain base-10 fractions (like 0.1 or 0.2),
small rounding errors occur.
While tiny, these errors compound when repeatedly added over thousands of simulated
operational timesteps

A classic example is the 0.1 + 0.2 problem

In [1]:
a = 0.1
b = 0.2
result = a + b

print(result)
print(result == 0.3)

0.30000000000000004
False


The expected answer is 0.3 and the conditional statement would print `True` but that is
not the case because neither 0.1 not 0.2 can be stored exactly in binary. When added, the
tiny errors accumulate, producing a result slightly above 0.3

How the error happens can be shown by looking at how the nubers are stored in memory.

In [2]:
from decimal import Decimal

print(Decimal(0.1))
print(Decimal(0.2))
print(Decimal(0.3))

0.1000000000000000055511151231257827021181583404541015625
0.200000000000000011102230246251565404236316680908203125
0.299999999999999988897769753748434595763683319091796875


Good scientific computing systems acknowledge them and use tolerances for comparisons
and monitors whether errors accumulate to a level that affects decisions